# ☀️ Utility-Scale Solar Power Forecasting System: Plant 1 Ingestion Matrix

This operational architecture implements a high-integrity predictive pipeline for an industrial-scale Photovoltaic (PV) asset. Before designing our feature space, this section executes the primary **Data Ingestion Sequence** and audits the baseline structural dimensions of the raw streaming telemetry.

### ⚙️ Domain Framework & Analytical Blueprint:
1. **Telemetry Stream A (Power Generation):** Ingests raw physical output metrics spanning multiple inverter sub-assets. Key variables map the transition from panel-side Direct Current ($DC$) to grid-synchronized Alternating Current ($AC$).
2. **Telemetry Stream B (Meteorological Sensors):** Captures ambient thermal parameters, active photovoltaic module surface temperatures, and plane-of-array solar irradiation metrics ($W/m^2$).
3. **Enterprise Ingestion Logging:** Replaces loose terminal prints with a synchronized, production-grade logging engine configuration to establish runtime traceability from step zero.

In [ ]:
# ==============================================================================
# SECTION 1: SYSTEM INGESTION & STRUCTURAL INTEGRITY AUDIT (ENTERPRISE LOGGING)
# ==============================================================================
import os
import logging
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score

# 1. Initialize Root Pipeline Logging Architecture
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("IngestionPipeline")

# 2. Establish Secure Path Routing Safeguards
gen_path = '../data/raw/Plant_1_Generation_Data.csv' if os.path.exists('../data/raw/Plant_1_Generation_Data.csv') else 'Plant_1_Generation_Data.csv'
weather_path = '../data/raw/Plant_1_Weather_Sensor_Data.csv' if os.path.exists('../data/raw/Plant_1_Weather_Sensor_Data.csv') else 'Plant_1_Weather_Sensor_Data.csv'

logger.info("Initializing raw Plant 1 telemetry datasets ingestion matrix...")
plant1_gen_df = pd.read_csv(gen_path)
plant1_weather_df = pd.read_csv(weather_path)

# 3. Output Dimensions Audit via System Logging Constraints
logger.info(f"Ingestion verified. Generation Data Matrix Shape: {plant1_gen_df.shape[0]:,} rows × {plant1_gen_df.shape[1]} columns")
logger.info(f"Ingestion verified. Weather Sensor Data Matrix Shape: {plant1_weather_df.shape[0]:,} rows × {plant1_weather_df.shape[1]} columns")

print("\n" + "="*80)
print(" PLANT 1 TELEMETRY COMPLIANCE SHIELD: INITIAL SCHEMA HEAD SAMPLE")
print("==============================================================================")
# Display clean structured head records without index column noise
print(plant1_gen_df.head(5).to_string(index=False))
print("==============================================================================")

## 🗂️ Plant 1: Structural Schema Discovery & Temporal Boundary Audit

This section executes a comprehensive structural diagnostic on the newly ingested Plant 1 data structures. We inspect specific feature allocations, evaluate tensor datatype health across columns, and securely lock the temporal boundaries (Start/End operation stamps) of our time-series domain.

### 📐 Strategic Quality Measures Applied:
1. **Schema Integrity Screening:** Utilizing structural tracking metrics (`.info()`) to audit missingness and cross-examine numeric features before running transformations.
2. **Deterministic Datetime Serialization:** Explicitly processing datetime strings via a strict format mask (`%d-%m-%Y %H:%M`). This ensures memory alignment and permanently blocks future localization warnings.
3. **Temporal Invariant Verification:** Extracting global chronological minima and maxima to verify alignment with the expected 34-day operational boundary.

In [ ]:
# ==============================================================================
# SECTION 2: SCHEMA CAPACITIES & TEMPORAL BOUNDARY VERIFICATION
# ==============================================================================

# Access the synchronized root logging manager instance
logger = logging.getLogger("IngestionPipeline")

print("\n" + "="*80)
print(" PLANT 1 WEATHER TELEMETRY: INITIAL SCHEMA HEAD SAMPLE")
print("==============================================================================")
# Display clean weather head records without index column noise
print(plant1_weather_df.head(5).to_string(index=False))
print("==============================================================================\n")

# 1. Audit unique physical assets registered inside the data stream
inverter_count = plant1_gen_df['SOURCE_KEY'].nunique()
logger.info(f"Structural Audit: Discovered {inverter_count} unique active inverter sub-assets (SOURCE_KEY) inside Plant 1.")

# 2. Render structured system datatype grids cleanly
print("\n" + "-"*40 + " [GENERATION MATRIX METADATA] " + "-"*40)
plant1_gen_df.info()

print("\n" + "-"*40 + " [WEATHER MATRIX METADATA]    " + "-"*40)
plant1_weather_df.info()

# 3. Execute deterministic datetime conversion with protective string formatting masks
logger.info("Executing format-specified datetime serialization over text trackers...")
plant1_gen_df['DATE_TIME'] = pd.to_datetime(plant1_gen_df['DATE_TIME'], format='%d-%m-%Y %H:%M')
plant1_weather_df['DATE_TIME'] = pd.to_datetime(plant1_weather_df['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

# 4. Extract explicit time-series operational limits
start_boundary = plant1_gen_df['DATE_TIME'].min()
end_boundary = plant1_gen_df['DATE_TIME'].max()

print("\n" + "="*80)
print(" CHRONOLOGICAL OPERATION TIMELINE AUDIT SIGN-OFF")
print("==============================================================================")
print(f"Plant 1 Operational Footprint Start Time: {start_boundary}")
print(f"Plant 1 Operational Footprint End Time  : {end_boundary}")
print("==============================================================================")
logger.info("Temporal boundary synchronization complete. Data schemas validated successfully.")

## ☀️ Plant 1: Chronological Diurnal Power Profile Analysis (DC vs AC)

To verify the physical operational metrics and capture the baseline solar zenith curve, this section isolates a specific calendar day (`2020-05-15`). We aggregate the cross-sectional mean generation path across all active inverters to cross-examine panel-side Direct Current ($DC$) against inverter-side Alternating Current ($AC$).

### 📐 Strategic Engineering Rules Applied:
1. **Dynamic Temporal Filtering:** Utilizing vectorized datetime date-matching constraints to extract the exact 24-hour target window safely.
2. **Cross-Sectional Smoothing:** Grouping by time tickers to calculate the cross-sectional mathematical mean, effectively attenuating localized sub-asset noise or dropout anomalies.
3. **Advanced Axis Formatting:** Integrating a strict matplotlib chronological date-locator strategy to prevent time label collisions or overlapping pixel text on the visual canvas.

In [ ]:
# ==============================================================================
# SECTION 3: CHRONOLOGICAL DIURNAL POWER PROFILE VISUALIZATION
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")

# Specify the target operational date to analyze (Dynamic intercept parameter)
target_date = '2020-05-15'
logger.info(f"Isolating 24-hour chronological generation stream for target date: {target_date}")

# 2. Filter the dataset to include strictly the target execution date window
day_df = plant1_gen_df[plant1_gen_df['DATE_TIME'].dt.date == pd.to_datetime(target_date).date()].copy()

# 3. Calculate cross-sectional mean profiles across all 22 available inverter assets
hourly_avg = day_df.groupby('DATE_TIME')[['DC_POWER', 'AC_POWER']].mean().reset_index()

# 4. Initialize Plot Canvas Framework
plt.figure(figsize=(14, 6))

# Plot the physical generation trajectories symmetrically
plt.plot(hourly_avg['DATE_TIME'], hourly_avg['DC_POWER'], label='DC Power Output (Panel Side)', color='orange', lw=2.5)
plt.plot(hourly_avg['DATE_TIME'], hourly_avg['AC_POWER'], label='AC Power Output (Grid Side)', color='blue', lw=2.0, linestyle='--')

# 5. FIX: Apply Explicit Chronological Locators to Prevent Label Collisions and UserWarnings
ax = plt.gca()
# Automatically space ticks every 2 hours across the diurnal timeline
ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
# Enforce an enterprise-grade military 24-hour format string representation
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.title(f"Plant 1: Overlaid Chronological Diurnal Power Profile Wave ({target_date})", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day (24-Hour Cycle)", fontsize=11)
plt.ylabel("Consolidated Power Metrics (kW)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

logger.info("Diurnal power profile curve rendered successfully. Physical boundary audit closed.")

## 📊 Plant 1: Daily Accumulated Energy Yield Distribution Matrix

To evaluate the operational volume and macro efficiency transformations of the asset across its full lifespan, this section aggregates the absolute daily maximum boundaries of the cumulative metrics (`DAILY_YIELD`). This isolates the net energy volume ($kWh$) generated from dawn to dusk for each of the 34 monitoring days.

### 📐 Strategic Quality Measures Applied:
1. **Vectorized Temporal Isolation:** Extracting native date profiles from synchronized datetime indexes to anchor day-level groupings safely.
2. **Boundary Frontier Extraction:** Capturing the true mathematical maximum (`.max()`) to ensure the absolute end-of-day total reading is extracted without tracking errors.
3. **Optimized X-Axis Tick Density:** Programmatically subsampling chronological categorical dates to enforce spacing parameters, guaranteeing crisp layout presentation without horizontal character smashing.

In [ ]:
# ==============================================================================
# SECTION 4: DAILY ACCUMULATED YIELD FRONTIER VISUALIZATION
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Compiling daily absolute maximum boundaries for cumulative energy yield vectors.")

# Create a temporary localized clean string column representing the calendar date
plant1_gen_df['DATE_STR'] = plant1_gen_df['DATE_TIME'].dt.strftime('%Y-%m-%d')

# 2. Calculate the maximum DAILY_YIELD for each distinct date (capturing the final end-of-day state)
daily_yield_max = plant1_gen_df.groupby('DATE_STR')['DAILY_YIELD'].max().reset_index()

# 3. Initialize Visual Plot Grid
plt.figure(figsize=(15, 6))

# Render clean, unified teal bar chart representations
ax = sns.barplot(data=daily_yield_max, x='DATE_STR', y='DAILY_YIELD', color='teal', edgecolor='darkslategray', alpha=0.85)

# 4. FIX: Apply Explicit Fixed Ticks and Subsample Density to Prevent Overlapping text block collisions
x_indices = np.arange(len(daily_yield_max))
ax.set_xticks(x_indices)

# Optimize text layout: Show tick labels every 2 days instead of smashing all 34 dates horizontally
tick_labels = [label if i % 2 == 0 else '' for i, label in enumerate(daily_yield_max['DATE_STR'])]
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=9)

plt.title("Plant 1: 34-Day Maximum Aggregated Daily Energy Yield Profile (kWh)", fontsize=14, fontweight='bold')
plt.xlabel("Calendar Operation Date Target", fontsize=11)
plt.ylabel("Accumulated Daily Yield Capacity (kWh)", fontsize=11)
plt.grid(True, axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

logger.info("Daily cumulative yield bar dashboard compiled silently with zero warnings.")

## 📈 Plant 1: Plant-Wide Total Cumulative Yield Escalation Trajectory

To evaluate the long-term energy compounding velocity of the asset across its full monitoring lifespan, this section consolidates the lifetime integrated registry metrics (`TOTAL_YIELD`). We aggregate the cross-sectional sum across all active inverters per timestamp to map the plant-wide cumulative growth vector.

### 📐 Strategic Engineering Measures Applied:
1. **Cross-Sectional Volumetric Summation:** Aggregating asset counters to track the consolidated macro growth trajectory of the entire infrastructure.
2. **Scientific Notation Suppression:** Deploying a custom numeric `FuncFormatter` to eliminate exponential scaling tags (e.g., $1e7$), guaranteeing executive-ready integers on the vertical axis layer.
3. **Chronological Locator Masking:** Leveraging globally imported `mdates` locators to structure the 34-day operational boundary into clean, bi-weekly interval ticks symmetrically.

In [ ]:
# ==============================================================================
# SECTION 5: TOTAL CUMULATIVE YIELD ESCALATION TRAJECTORY
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Aggregating cross-sectional sum for lifetime integrated total yield counters.")

# Calculate the total cumulative yield across all inverters for each distinct timestamp
total_yield_trend = plant1_gen_df.groupby('DATE_TIME')['TOTAL_YIELD'].sum().reset_index()

# 2. Initialize Visual Plot Framework
plt.figure(figsize=(14, 6))
plt.plot(total_yield_trend['DATE_TIME'], total_yield_trend['TOTAL_YIELD'], color='darkred', lw=2.5)

# 3. FIX: Apply Explicit Chronological Locators and Formatter to Prevent Overlapping text block collisions
ax = plt.gca()
# Automatically space major ticks every 4 days across the 34-day continuous timeline
ax.xaxis.set_major_locator(mdates.DayLocator(interval=4))
# Enforce an enterprise-grade standard date format representation (Year-Month-Day)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

# 4. Suppress scientific notation on the vertical axis for executive readability compliance
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

plt.title("Plant 1: Consolidated Total Lifetime Energy Yield Escalation Trajectory (kWh)", fontsize=14, fontweight='bold')
plt.xlabel("Calendar Operation Timeline (34-Day Continuous Window)", fontsize=11)
plt.ylabel("Consolidated Lifetime Yield Volume (Cumulative kWh)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

logger.info("Total lifetime trajectory visualization completed with global imports synchronized.")

## 🌡️ Plant 1: Meteorological Context Analytics (Thermal Disparity & Irradiation Framework)

To build a rigorous causal framework for our machine learning model, this section analyzes the environmental independent variables ($X$). We first map the 34-day continuous thermal trends—overlapping raw atmospheric heat ($Ambient\_Temperature$) against panel surface friction heat ($Module\_Temperature$). We then isolate a 24-hour window (`2020-05-15`) to extract the continuous solar influx curve ($Irradiation$).

### 📐 Strategic Quality Measures Applied:
1. **Dynamic Thermal Overlays:** Mapping the ambient air (Blue) against panel surface module heat (Red) to understand thermal efficiency degradation rules.
2. **High-Frequency Axis Deflation:** Integrating dynamic `DayLocator` constraints to prevent time stamp text collisions over the 34-day continuous vector line.
3. **Diurnal Solar Zenith Verification:** Plotting the 24-hour irradiation path using explicit `HourLocator` intervals, ensuring a crisp, executive-ready view of the plant's core energy input wave.

In [ ]:
# ==============================================================================
# SECTION 6: METEOROLOGICAL LIFESPAN TRENDS & DIURNAL IRRADIATION WAVE
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Rendering 34-day continuous atmospheric thermal disparity profiles.")

# --- CHART 1: 34-DAY TEMPERATURE TREND (AMBIENT VS MODULE) ---
plt.figure(figsize=(15, 6))

# Plot the ambient and module temperatures overlapping each other
plt.plot(plant1_weather_df['DATE_TIME'], plant1_weather_df['AMBIENT_TEMPERATURE'], label='Ambient Air Temperature', color='blue', alpha=0.6, lw=1.5)
plt.plot(plant1_weather_df['DATE_TIME'], plant1_weather_df['MODULE_TEMPERATURE'], label='Module Surface Temperature', color='red', alpha=0.4, lw=1.5)

# FIX: Apply Explicit Chronological Locators to Prevent Overlapping Character Smudging
ax1 = plt.gca()
ax1.xaxis.set_major_locator(mdates.DayLocator(interval=4))  # Space ticks every 4 days cleanly
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

plt.title("Plant 1: 34-Day Continuous Meteorological Temperature Trend (Ambient vs Module)", fontsize=14, fontweight='bold')
plt.xlabel("Calendar Operation Timeline (34-Day Continuous Window)", fontsize=11)
plt.ylabel("Atmospheric Thermal Metrics (°C)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# --- CHART 2: 24-HOUR SOLAR IRRADIATION FLASHPOINT ---
target_date_weather = '2020-05-15'
logger.info(f"Isolating 24-hour solar irradiation trajectory for target date: {target_date_weather}")

# Filter the dataset to include strictly the specified target operational date window
day_weather_df = plant1_weather_df[plant1_weather_df['DATE_TIME'].dt.date == pd.to_datetime(target_date_weather).date()].copy()

plt.figure(figsize=(14, 5))
plt.plot(day_weather_df['DATE_TIME'], day_weather_df['IRRADIATION'], color='gold', lw=2.5, label='Solar Irradiation Intensity')

# FIX: Apply Explicit Chronological Locators for the Day-Level Tickers
ax2 = plt.gca()
ax2.xaxis.set_major_locator(mdates.HourLocator(interval=2))  # Space major ticks every 2 hours beautifully
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.title(f"Plant 1: 24-Hour Continuous Solar Irradiation Wave ({target_date_weather})", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day (24-Hour Cycle)", fontsize=11)
plt.ylabel("Solar Irradiation Intensity ($W/m^2$)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

logger.info("Meteorological context visualizations generated successfully. Environmental audit closed.")

## 📊 Plant 1: Cross-Sectional Daily Aggregation & System Anomaly Screening

Before engineering predictive model features, we execute a multi-metric numerical audit on `DAILY_YIELD` and `DC_POWER`. By collapsing the high-frequency time-series array into descriptive statistical vectors ($\text{Max}, \text{Min}, \text{Mean}, \text{Median}, \text{Count}$) per calendar date, we can immediately isolate hardware dropouts, sensor clipping, or telemetric frame data loss.

### 📐 Strategic Quality Measures Applied:
1. **Multi-Vector Structural Descriptors:** Combining central tendency metrics ($\text{Mean}, \text{Median}$) with extreme boundary frontiers ($\text{Max}, \text{Min}$) to detect skewed data shapes or artificial floor states.
2. **Telemetry Count Monitoring:** Auditing the absolute data point volume (`count`) per day to cross-examine network stability and verify whether any target operation slots suffered severe packet dropouts.
3. **Multi-Level Column Demolition:** Linearly flattening pandas multi-index headers into crisp, enterprise-grade snake_case labels for optimal upstream feature extraction compliance.

In [ ]:
# ==============================================================================
# SECTION 7: CROSS-SECTIONAL DAILY AGGREGATION & ANOMALY SCREENING
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Compiling multi-vector cross-sectional summary matrices for power and yield features.")

# Ensure we rely on our synchronized local date tracker format string
if 'DATE_STR' not in plant1_gen_df.columns:
    plant1_gen_df['DATE_STR'] = plant1_gen_df['DATE_TIME'].dt.strftime('%Y-%m-%d')

# 2. Execute Multi-Metric Structural Aggregations Over Calendar Targets
daily_summary = plant1_gen_df.groupby('DATE_STR').agg({
    'DAILY_YIELD': ['max', 'min', 'mean', 'median', 'count'],
    'DC_POWER': ['max', 'min', 'mean', 'median']
}).reset_index()

# 3. Flatten the Pandas Multi-Level Hierarchical Columns for High Readability
daily_summary.columns = [
    'date', 
    'yield_max', 'yield_min', 'yield_mean', 'yield_median', 'telemetry_count',
    'dc_max', 'dc_min', 'dc_mean', 'dc_median'
]

# Sort chronologically to preserve execution timeline flow safely
daily_summary = daily_summary.sort_values('date').reset_index(drop=True)

print("\n" + "="*95)
print(" PLANT 1 OPERATIONAL INTEGRITY REPORT: 34-DAY MULTI-VECTOR SUMMARY CAPACITY")
print("===============================================================================================")
# Display the entire 34-day summary data frame cleanly without index or formatting noise
pd.set_option('display.max_rows', 50)
print(daily_summary.to_string(index=False, formatters={
    'yield_max': '{:,.1f}'.format, 'yield_min': '{:,.1f}'.format, 'yield_mean': '{:,.1f}'.format, 'yield_median': '{:,.1f}'.format,
    'dc_max': '{:,.2f}'.format, 'dc_min': '{:,.2f}'.format, 'dc_mean': '{:,.2f}'.format, 'dc_median': '{:,.2f}'.format,
    'telemetry_count': '{:,}'.format
}))
print("===============================================================================================")

logger.info(f"Descriptive screening matrix compiled successfully. Total timeline checked: {len(daily_summary)} operational days.")

## 📊 Plant 1: 34-Day Multi-Facet Diurnal DC Power Distribution Audit

To screen chronological operational consistency and isolate localized equipment dropouts across the entire monitoring lifespan, this section constructs a **Multi-Facet Subplot Grid**. We map the high-frequency diurnal $DC\_POWER$ footprint as an unlinked dot distribution canvas for each of the 34 operational calendar days independently.

### 📐 Strategic Quality Measures Applied:
1. **Defensive Array Flattening:** Isolating flattened subplot axes into a distinct variable workspace (`axes_flat`) to preserve the structural dimensions of the original layout matrix, preventing runtime loop errors.
2. **Unlinked Token Densities:** Eliminating continuous line interpolation (`linestyle='None'`) to map raw telemetric points as granular scatter footprints, exposing exact temporal data gaps.
3. **Multi-Facet Ticker Anchoring:** Programmatically routing time-stamp indices via central coordinate locators, ensuring that military 24-hour markers ($00:00$ to $20:00$) align across all 34 grids without horizontal character smudging.

In [ ]:
# ==============================================================================
# SECTION 8: 34-DAY MULTI-FACET DIURNAL DC POWER DOT DISTRIBUTION
# ==============================================================================


# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initializing multi-facet grid canvas for 34-day diurnal power distribution auditing.")

# 2. Create clean staging copies to avoid altering original raw dataframes
p1_gen_clean = plant1_gen_df.copy()

# Enforce explicit datetime data types tracking format stability
p1_gen_clean['DATE_TIME'] = pd.to_datetime(p1_gen_clean['DATE_TIME'])
p1_gen_clean['DATE_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%Y-%m-%d')
p1_gen_clean['TIME_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%H:%M')

# 3. Extract a unique sorted list of all 34 dates from the engineered array
unique_dates = sorted(p1_gen_clean['DATE_STR'].unique())
total_plots = len(unique_dates)

cols = 5
rows = math.ceil(total_plots / cols)

# 4. Initialize the subplot grid with shared axes for absolute comparison scaling
fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 3.5), sharex=True, sharey=True)

# FIX: Isolate flattened variable into axes_flat to protect the original multidimensional array layout
axes_flat = axes.flatten()

# 5. Loop through each date and plot its 24-hour DC_POWER tracking dots
for i, current_date in enumerate(unique_dates):
    day_df = p1_gen_clean[p1_gen_clean['DATE_STR'] == current_date].sort_values('DATE_TIME')
    ax = axes_flat[i]
    
    # Plot as individual dots using marker='.' and removing the connecting line
    ax.plot(day_df['TIME_STR'], day_df['DC_POWER'], marker='.', linestyle='None', color='royalblue', alpha=0.6, markersize=4)
    
    ax.set_title(current_date, fontsize=10, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # FIX: Secure chronological tick density allocation inside the shared layout loop safely
    # Generate explicit integer indexes for 15-minute intervals (Every 4 hours = 16 steps of 15 mins)
    target_indices = np.arange(0, len(day_df['TIME_STR'].unique()), 16)
    ax.set_xticks(target_indices)
    ax.set_xticklabels(sorted(day_df['TIME_STR'].unique())[::16], rotation=45, fontsize=8)

# 6. Remove empty subplot grid containers safely using the isolated dimensional framework
for j in range(total_plots, len(axes_flat)):
    fig.delaxes(axes_flat[j])

# Configure global super-labels across the layout canvas canvas
fig.suptitle("Plant 1: 34-Day Isolated Grids of Daily 24-Hour DC Power Dot Distribution (kW)", fontsize=16, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()

logger.info("Multi-facet scatter grid rendered successfully with dimensional invariants protected.")

## 📊 Plant 1: 34-Day Multi-Facet Diurnal Cumulative Yield Distribution Audit

To screen chronological accumulation integrity and isolate physical telemetry counting anomalies across the monitoring lifespan, this section constructs a **Multi-Facet Subplot Grid**. We map the high-frequency diurnal $DAILY\_YIELD$ plant-wide summation as an unlinked dot trajectory canvas for each of the 34 operational calendar days independently.

### 📐 Strategic Quality Measures Applied:
1. **Defensive Array Flattening:** Isolating flattened subplot axes into a distinct variable workspace (`axes_flat`) to preserve the structural dimensions of the original layout matrix, preventing runtime loop index exceptions.
2. **Volumetric Plant Aggregation:** Summing individual inverter registers per timestamp to audit the consolidated macro charging curve ($S\text{-curve}$) of the entire infrastructure symmetrically.
3. **Multi-Facet Ticker Anchoring:** Programmatically routing time-stamp indices via central coordinate locators, ensuring that military 24-hour markers ($00:00$ to $20:00$) align across all 34 grids without horizontal character smudging.

In [ ]:
# ==============================================================================
# SECTION 9: 34-DAY MULTI-FACET DIURNAL CUMULATIVE YIELD DISTRIBUTION
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initializing multi-facet grid canvas for 34-day plant-wide cumulative yield auditing.")

# 2. Create a safe staging copy to protect original data and avoid altering the raw dataframe
p1_gen_clean = plant1_gen_df.copy()

# Enforce explicit datetime data types tracking format stability
p1_gen_clean['DATE_TIME'] = pd.to_datetime(p1_gen_clean['DATE_TIME'])
p1_gen_clean['DATE_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%Y-%m-%d')
p1_gen_clean['TIME_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%H:%M')

# 3. Group by Date and Time to calculate the SUM of DAILY_YIELD across all 22 inverters
plant_total_yield = p1_gen_clean.groupby(['DATE_STR', 'TIME_STR', 'DATE_TIME'])['DAILY_YIELD'].sum().reset_index()

# 4. Extract the unique sorted list of all 34 dates
unique_dates = sorted(plant_total_yield['DATE_STR'].unique())
total_plots = len(unique_dates)

cols = 5
rows = math.ceil(total_plots / cols)

# 5. Initialize the subplot grid with shared axes for absolute comparison scaling
fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 3.5), sharex=True, sharey=True)

# FIX: Isolate flattened variable into axes_flat to protect the original multidimensional array layout
axes_flat = axes.flatten()

# 6. Loop through each date and plot the plant's total cumulative yield as dots
for i, current_date in enumerate(unique_dates):
    day_df = plant_total_yield[plant_total_yield['DATE_STR'] == current_date].sort_values('DATE_TIME')
    ax = axes_flat[i]
    
    # Plotting as scatter dots (marker='.') without any connecting lines
    ax.plot(day_df['TIME_STR'], day_df['DAILY_YIELD'], marker='.', linestyle='None', color='teal', alpha=0.8, markersize=5)
    
    ax.set_title(current_date, fontsize=10, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # FIX: Secure chronological tick density allocation inside the shared layout loop safely
    # Generate explicit integer indexes for 15-minute intervals (Every 4 hours = 16 steps of 15 mins)
    target_indices = np.arange(0, len(day_df['TIME_STR'].unique()), 16)
    ax.set_xticks(target_indices)
    ax.set_xticklabels(sorted(day_df['TIME_STR'].unique())[::16], rotation=45, fontsize=8)

# 7. Remove empty subplot grid containers safely using the isolated dimensional framework
for j in range(total_plots, len(axes_flat)):
    fig.delaxes(axes_flat[j])

# Configure global super-labels across the layout canvas
fig.suptitle("Plant 1: 34-Day Isolated Grids of Daily 24-Hour Total Plant Yield Distribution (Cumulative kWh)", fontsize=16, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()

logger.info("Multi-facet yield trajectory grid rendered successfully with dimensional invariants protected.")

## 🔍 Plant 1: Automated Data Quality Auditing & Telemetry Outage Detection

To guarantee the structural security of our machine learning features space, this section operationalizes an automated **Data Quality Audit Pipeline**. We compute daily operational boundaries and evaluate each calendar date against strict data infrastructure metrics to flag hardware dropouts or telemetric packet data loss.

### 📐 Automated Quality Gates Applied:
1. **Telemetry Point Constraint (`telemetry_count == 2,112`):** Validates frame completeness. Any day dropping below 2,112 points (22 inverters $\times$ 96 logs per day) triggers a **Critical Outage Alert**.
2. **Volumetric Floor Constraint (`yield_max < 4,000`):** Screens for underperformance or severe maintenance grid shutdowns on days where data packet collection was otherwise complete, triggering a **Performance Warning**.
3. **Multi-Level Compliance Logging:** Integrates with our standardized log manager to report system alerts dynamically before exporting clean arrays.

In [ ]:
# ==============================================================================
# SECTION 10: AUTOMATED DATA QUALITY AUDITING & INFRASTRUCTURE CHECKS
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initiating automated telemetric frame data quality checks over Plant 1 arrays.")

# Create clean staging copies to avoid altering original raw dataframes
p1_gen_clean = plant1_gen_df.copy()

# Ensure chronological format string is populated
if 'DATE_STR' not in p1_gen_clean.columns:
    p1_gen_clean['DATE_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%Y-%m-%d')

# 2. Group by Date to Compute Operational Metrics Frontiers
cleaning_check = p1_gen_clean.groupby('DATE_STR').agg({
    'DAILY_YIELD': 'max',
    'DC_POWER': 'max',
    'DATE_TIME': 'count'
}).reset_index()

# Flatten headers into strict lowercase snake_case for software compliance rules
cleaning_check.columns = ['date', 'yield_max', 'dc_max', 'telemetry_count']

# Sort chronologically to preserve timeline tracking integrity
cleaning_check = cleaning_check.sort_values('date').reset_index(drop=True)

# 3. Apply Quality Gates to Filter Structural System Anomalies
# Gate A: Packet Dropouts Screening (Theoretical full day capacity = 2,112 rows)
flagged_outages = cleaning_check[cleaning_check['telemetry_count'] < 2112]

# Gate B: Low Performance / Grid Shutdown Screening on complete telemetry days
flagged_low_perf = cleaning_check[(cleaning_check['yield_max'] < 4000) & (cleaning_check['telemetry_count'] == 2112)]

# 4. Render Enterprise Structured Integrity Exception Reports
print("\n" + "="*85)
print(" SYSTEM INTEGRITY ALERT A: TELEMETRY FRAME DROPOUTS DETECTED (CRITICAL)")
print("=====================================================================================")
if not flagged_outages.empty:
    logger.error(f"Data density breach: Found {len(flagged_outages)} operational days with missing rows.")
    print(flagged_outages.to_string(index=False, formatters={
        'yield_max': '{:,.1f}'.format, 'dc_max': '{:,.2f}'.format, 'telemetry_count': '{:,}'.format
    }))
else:
    print("Zero runtime telemetric row dropouts detected. All days meet the 2,112 capacity frontier.")
print("=====================================================================================\n")

print("="*85)
print(" SYSTEM INTEGRITY ALERT B: OPERATIONAL UNDERPERFORMANCE / DROPS DETECTED (WARNING)")
print("=====================================================================================")
if not flagged_low_perf.empty:
    logger.warning(f"Capacity drift anomaly: Found {len(flagged_low_perf)} days violating thermal floors.")
    print(flagged_low_perf.to_string(index=False, formatters={
        'yield_max': '{:,.1f}'.format, 'dc_max': '{:,.2f}'.format, 'telemetry_count': '{:,}'.format
    }))
else:
    print("Zero structural yield dropouts detected. All full telemetry days match healthy metrics.")
print("=====================================================================================")

logger.info("Data quality auditing sequence executed. Anomaly verification matrices logged.")

## 🔍 Plant 1: Peak-Hour Hardware Anomaly Screening (Noontime Culprit Hunting)

To isolate true electrical component failures from temporary environmental shadows, this section operationalizes a **Peak-Hour Anomaly Screening Pipeline**. We focus strictly on the peak solar window ($\text{11:00 AM} \sim \text{03:00 PM}$) and evaluate high-frequency $DC\_POWER$ records to pinpoint individual inverter assets suffering from prolonged grid dropouts or hardware shutdowns.

### 📐 Strategic Engineering Rules Applied:
1. **Targeted Peak Subsampling:** Constraining the evaluation timeline to peak radiation hours to maximize signal-to-noise ratio when checking asset physical health.
2. **Dynamic Zero-Power Aggregation:** Computing the exact duration (number of 15-minute intervals) where panel-side output registered exactly $0.0\text{ kW}$ despite maximum solar zenith influx.
3. **Hardware Failure Thresholding (`zero_count > 4`):** Setting a strict 1-hour temporal threshold to safely filter out short-term cloud movements while isolating chronic hardware outages.
4. **Lowercase Standardization Compliance:** Mapping output labels to uniform lowercase snake_case structures (`inverter_id`, `zero_count_noon`) for upstream enterprise script integration.

In [ ]:
# ==============================================================================
# SECTION 11: PEAK-HOUR HARDWARE ANomaly SCREENING (CULPRIT HUNTING)
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initializing high-frequency hardware failure screening during peak solar operation hours.")

# Create clean staging copies to avoid altering original raw dataframes
p1_gen_clean = plant1_gen_df.copy()

# Enforce explicit datetime data types tracking format stability
p1_gen_clean['DATE_TIME'] = pd.to_datetime(p1_gen_clean['DATE_TIME'])
p1_gen_clean['DATE_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%Y-%m-%d')
p1_gen_clean['TIME_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%H:%M')

# 2. Filter data strictly for peak operational hours (11:00 AM - 03:00 PM)
peak_df = p1_gen_clean[(p1_gen_clean['TIME_STR'] >= '11:00') & (p1_gen_clean['TIME_STR'] <= '15:00')].copy()

# 3. Aggregate noontime dropout counters across calendar targets and sub-assets
culprit_hunting = peak_df.groupby(['DATE_STR', 'SOURCE_KEY']).agg({
    'DC_POWER': lambda x: (x == 0.0).sum()
}).reset_index()

# Flatten headers into strict lowercase snake_case for software compliance rules
culprit_hunting.columns = ['date', 'inverter_id', 'zero_count_noon']

# 4. Filter for specific assets dead for more than 4 steps (exceeding 1 continuous hour)
detected_culprits = culprit_hunting[culprit_hunting['zero_count_noon'] > 4].copy()

# Sort chronologically and by asset key to preserve visual compliance flow safely
detected_culprits = detected_culprits.sort_values(['date', 'inverter_id']).reset_index(drop=True)

# 5. Render Enterprise Structured Integrity Exception Reports
print("\n" + "="*85)
print(" SYSTEM HARDWARE EXCEPTION REPORT: NOONTIME INVERTER DROPOUTS DETECTED")
print("=====================================================================================")
if not detected_culprits.empty:
    logger.error(f"Hardware anomaly alert: Found {len(detected_culprits)} sub-asset dropout events exceeding safety thresholds.")
    print(detected_culprits.to_string(index=False, formatters={
        'zero_count_noon': '{:,}'.format
    }))
else:
    print("Zero hardware dropouts detected during peak solar windows. All sub-assets operating within safe parameters.")
print("=====================================================================================")

logger.info("Hardware failure screening sequence finalized. Telemetry exceptions logged.")

## 📊 Plant 1: Hardware Failure & Commercial Loss Executive Dashboard

To deliver a clear operational diagnosis to senior stakeholders, this section constructs a **$2 \times 2$ Executive Diagnostic Dashboard**. We cross-examine the thermodynamic root cause ($DC\_POWER$ dropouts against the historical median baseline) alongside its direct commercial financial impact ($DAILY\_YIELD$ stagnation) over two critical failure dates (`2020-06-07` and `2020-06-14`).

### 📐 Strategic Engineering Layout Rules Applied:
1. **Dynamic Baseline Benchmarking:** Computing the cross-sectional median profile across the entire asset grid to act as a robust un-skewed reference frame.
2. **Causal-To-Financial Mapping:** Assigning the upstream telemetry cause (Row 1) directly above the downstream business yield loss (Row 2) to maintain perfect linear diagnostic narrative flow.
3. **Synchronized Shared-Axis Anchoring:** Programmatically mapping integer coordinate indices (`np.arange`) across shared-column configurations, ensuring time labels ($00:00 \sim 22:00$) register flawlessly across all quadrants without clipping or rendering artifacts.

In [ ]:
# ==============================================================================
# SECTION 12: EXECUTIVE HARDWARE FAILURE DIAGNOSTIC DASHBOARD
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Assembling 2x2 executive hardware failure and commercial loss validation dashboard.")

# Create clean staging copies to avoid altering original raw dataframes
p1_gen_clean = plant1_gen_df.copy()

# Enforce explicit datetime data types tracking format stability
p1_gen_clean['DATE_TIME'] = pd.to_datetime(p1_gen_clean['DATE_TIME'])
p1_gen_clean['DATE_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%Y-%m-%d')
p1_gen_clean['TIME_STR'] = p1_gen_clean['DATE_TIME'].dt.strftime('%H:%M')

# 2. Extract Healthy Plant Operational Benchmarks (Using Median to protect against skewness)
median_dc = p1_gen_clean.groupby('DATE_TIME')['DC_POWER'].median().reset_index().rename(columns={'DC_POWER': 'plant_dc_median'})
median_yield = p1_gen_clean.groupby('DATE_TIME')['DAILY_YIELD'].median().reset_index().rename(columns={'DAILY_YIELD': 'plant_yield_median'})

# Merge benchmarking frames into a consolidated diagnostic master matrix
dash_df = pd.merge(p1_gen_clean, median_dc, on='DATE_TIME', how='left')
dash_df = pd.merge(dash_df, median_yield, on='DATE_TIME', how='left').sort_values('DATE_TIME')

# 3. Define Explicit Analytical Checkpoints
target_dates = ['2020-06-07', '2020-06-14']
culprit_ids = ['1BY6WEcLGh8j5v7', 'bvBOhCH3iADSZry', 'wCURE6d3bPkepu2', 'z9Y9gH1T5YWrNuG']

# 4. Initialize Subplot Matrix Canvas Framework
fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharex='col', sharey='row')

# Calculate consistent tick spacing coordinates over the 24-hour time string vector (Every 2 hours = 8 steps of 15-min logs)
sample_timeline = sorted(dash_df['TIME_STR'].unique())
target_indices = np.arange(0, len(sample_timeline), 8)
target_labels = [sample_timeline[idx] for idx in target_indices]

# ==============================================================================
# ROW 1: DC POWER ANALYSIS (THE LOGICAL ROOT CAUSE)
# ==============================================================================
for idx, current_date in enumerate(target_dates):
    day_df = dash_df[dash_df['DATE_STR'] == current_date]
    culprits_day_df = day_df[day_df['SOURCE_KEY'].isin(culprit_ids)].sort_values('DATE_TIME')
    day_df = day_df.sort_values('DATE_TIME')
    ax = axes[0, idx]
    
    # Plot Healthy Plant Reference Frontier
    ax.plot(day_df['TIME_STR'], day_df['plant_dc_median'], color='gray', linestyle='--', lw=2, label='Healthy Plant Median Baseline')
    
    # Plot Anomalous Asset Scatter Profiles
    sns.scatterplot(
        data=culprits_day_df, x='TIME_STR', y='DC_POWER', hue='SOURCE_KEY', 
        palette='Set1', s=50, alpha=0.8, ax=ax
    )
    
    ax.set_title(f"Plant 1: DC Power Output Trend [{current_date}]", fontsize=13, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xlabel('') # Clear shared axis label for upper quadrant clarity
    
    if idx == 0:
        ax.set_ylabel("Direct Current Power Output (kW)", fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)

# ==============================================================================
# ROW 2: DAILY YIELD ANALYSIS (THE DOWNSTREAM COMMERCIAL LOSS)
# ==============================================================================
for idx, current_date in enumerate(target_dates):
    day_df = dash_df[dash_df['DATE_STR'] == current_date]
    culprits_day_df = day_df[day_df['SOURCE_KEY'].isin(culprit_ids)].sort_values('DATE_TIME')
    day_df = day_df.sort_values('DATE_TIME')
    ax = axes[1, idx]
    
    # Plot Healthy Plant Reference Frontier
    ax.plot(day_df['TIME_STR'], day_df['plant_yield_median'], color='gray', linestyle='--', lw=2, label='Healthy Plant Median Baseline')
    
    # Plot Anomalous Asset Scatter Profiles
    sns.scatterplot(
        data=culprits_day_df, x='TIME_STR', y='DAILY_YIELD', hue='SOURCE_KEY', 
        palette='Set1', s=50, alpha=0.8, ax=ax, legend=False
    )
    
    ax.set_title(f"Plant 1: Cumulative Energy Yield Trend [{current_date}]", fontsize=13, fontweight='bold')
    ax.set_xlabel("Time of Day (24-Hour Operational Cycle)", fontsize=11, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    
    if idx == 0:
        ax.set_ylabel("Accumulated Daily Yield Capacity (kWh)", fontsize=11, fontweight='bold')
        
    # FIX: Explicitly enforce anchored coordinate ticks across the active columns layout synchronously
    ax.set_xticks(target_indices)
    ax.set_xticklabels(target_labels, rotation=45, ha='right', fontsize=9)

# 5. Apply Global Dashboard Title Matrix styling
fig.suptitle(
    "Plant 1: Hardware Failure & Commercial Loss Executive Diagnostic Dashboard\nMidday Asset Thermal Overheating Analytics (June 07 vs. June 14)", 
    fontsize=18, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.show()

logger.info("Executive diagnostic validation multi-plot canvas generated successfully.")

## 📊 Plant 1: Global Meteorological Descriptive Statistics Overview

To evaluate the overall scaling boundaries and operational distribution profiles of our independent environmental variables ($X$), this section executes a global descriptive statistical audit (`.describe()`). We cross-examine the central tendencies and boundary frontiers of `AMBIENT_TEMPERATURE`, `MODULE_TEMPERATURE`, and `IRRADIATION` without any prior tracking filters applied.

### 📐 Strategic Engineering Measures Applied:
1. **Unfiltered Distribution Baseline:** Capturing the raw population parameters ($\text{Mean}, \text{Std}, \text{Min}, \text{Max}, \text{Quartiles}$) to establish an unmanipulated statistical anchor frame.
2. **Physical Boundary Sanity Check:** Reviewing minimum and maximum caps to immediately cross-verify telemetric register safety (e.g., ensuring negative irradiation or floating sensor anomalies are nonexistent).
3. **Structured Float Precision Controls:** Formatting the tabular terminal output with strict float precisions to optimize executive-ready scanning across all mathematical tiers.

In [ ]:
# ==============================================================================
# SECTION 13: GLOBAL METEOROLOGICAL DESCRIPTIVE STATISTICS OVERVIEW
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Compiling global population descriptive statistical metrics for weather telemetry features.")

# 2. Extract Descriptive Summaries Over Specified Atmospheric Tracks
weather_features = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
global_weather_summary = plant1_weather_df[weather_features].describe()

# 3. Render Enterprise Structured Integrity Ledger Reports
print("\n" + "="*85)
print(" PLANT 1 WEATHER TELEMETRY METRICS: RAW GLOBAL POPULATION SUMMARY LEDGER")
print("=====================================================================================")
# Enforce a highly standardized precision layout mask to maximize scanning comfort
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print(global_weather_summary)
print("=====================================================================================")
# Reset pandas float display format safely to avoid contaminating global styles downstream
pd.reset_option('display.float_format')

logger.info("Global weather description analysis complete. Baseline scaling constraints verified.")

## 🔍 Plant 1: Automated Weather Telemetry Quality Auditing & Outage Detection

To ensure the temporal completeness of our independent environmental feature tracks ($X$), this section operationalizes an automated **Meteorological Data Quality Audit**. We compute the daily transaction volume across our continuous weather streams and evaluate each calendar date against the theoretical capacity frontier ($\text{96 data points}$ per day) to isolate localized sensor dropouts or packet loss.

### 📐 Automated Quality Gates Applied:
1. **Weather Frame Constraint (`telemetry_count == 96`):** Validates logging frequency. Any day dropping below 96 logs (4 intervals per hour $\times$ 24 hours) immediately triggers a **Meteorological Incompleteness Alert**.
2. **Standardized Variable Namespace:** Converting tabular columns and outputs into strict lowercase snake_case formatting parameters (`date`, `telemetry_count`) for enterprise pipeline readiness.
3. **Defensive Staging Duplication:** Utilizing localized clean staging dataframes (`.copy()`) to process text transformations without contaminating the raw master source matrices.

In [ ]:
# ==============================================================================
# SECTION 14: AUTOMATED METEOROLOGICAL DATA QUALITY AUDITING
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initiating automated data density validation over weather sensor arrays.")

# 2. Initialize localized clean staging arrays to avoid mutating parent schemas
weather_check_df = plant1_weather_df[['DATE_TIME']].copy()

# Enforce explicit datetime type compliance and isolate calendar dates
weather_check_df['DATE_TIME'] = pd.to_datetime(weather_check_df['DATE_TIME'])
weather_check_df['DATE_STR'] = weather_check_df['DATE_TIME'].dt.strftime('%Y-%m-%d')

# 3. Compute cross-sectional row counts aggregated per operation calendar target
weather_daily_check = weather_check_df.groupby('DATE_STR').count().reset_index()

# Flatten columns into strict lowercase snake_case compliance styles
weather_daily_check.columns = ['date', 'telemetry_count']

# 4. Filter for specific dates violating the theoretical 96-point density floor
weather_outage_list = weather_daily_check[weather_daily_check['telemetry_count'] < 96].copy()

# Sort chronologically to preserve timeline tracking integrity safely
weather_outage_list = weather_outage_list.sort_values('date').reset_index(drop=True)

# 5. Render Enterprise Structured Integrity Exception Reports
print("\n" + "="*85)
print(" SYSTEM INTEGRITY ALERT C: METEOROLOGICAL SENSOR PACKET LOSS DETECTED")
print("=====================================================================================")
if not weather_outage_list.empty:
    logger.error(f"Meteorological density breach: Found {len(weather_outage_list)} operational days with missing weather logs.")
    print(weather_outage_list.to_string(index=False, formatters={
        'telemetry_count': '{:,}'.format
    }))
else:
    print("Perfect execution state. Zero missing weather sensor tracking slots discovered.")
print("=====================================================================================")

logger.info("Meteorological quality auditing sequence closed. Telemetry densities verified.")

# 🛠️ CRISP-DM Step 3: Data Preparation (Plant 1 Engineering Pipeline)

This structural block operationalizes the **Data Preparation Phase** under the industry-standard CRISP-DM framework. To build a robust, leak-free feature space for our downstream LightGBM forecasting engine, we isolate our parent data matrices and establish uniform, standardized datetime serialization metrics.

### ⚙️ Analytical Blueprint & Engineering Safeguards:
1. **Defensive Workspace Segregation (`.copy()`):** Instantiating isolated staging copies to ensure memory-level mutations remain local, completely protecting parent data integrity.
2. **Deterministic Temporal Alignment:** Standardizing text-based temporal entries into high-precision synchronized datetime registers across both generation and weather streams.
3. **Upstream Multi-Key Engineering:** Extracting uniform calendar date strings (`date_str`) and intra-day tickers (`time_str`) to facilitate seamless 1:1 relational joins and cross-sectional aggregations.

In [ ]:
# ==============================================================================
# SECTION 15: CRISP-DM STEP 3 - DATA PREPARATION (DATETIME STANDARDIZATION)
# ==============================================================================


# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Operationalizing CRISP-DM Step 3: Initializing Plant 1 data preparation pipeline.")

# 2. Create defensive localized clean staging copies to protect raw data frames
p1_gen_clean = plant1_gen_df.copy()
p1_weather_clean = plant1_weather_df.copy()

# 3. Enforce strict datetime serialization across dual streaming tracks
logger.info("Synchronizing temporal features space formats across generation and meteorological tracks.")
p1_gen_clean['DATE_TIME'] = pd.to_datetime(p1_gen_clean['DATE_TIME'])
p1_weather_clean['DATE_TIME'] = pd.to_datetime(p1_weather_clean['DATE_TIME'])

# 4. Extract uniform relational text trackers (Enforce clean lowercase snake_case naming conventions)
p1_gen_clean['date_str'] = p1_gen_clean['DATE_TIME'].dt.strftime('%Y-%m-%d')
p1_gen_clean['time_str'] = p1_gen_clean['DATE_TIME'].dt.strftime('%H:%M')

p1_weather_clean['date_str'] = p1_weather_clean['DATE_TIME'].dt.strftime('%Y-%m-%d')
p1_weather_clean['time_str'] = p1_weather_clean['DATE_TIME'].dt.strftime('%H:%M')

logger.info("Plant 1 data preparation timeline initialized. Relational string trackers generated successfully.")

## 🛠️ CRISP-DM Step 3: End-to-End Data Cleansing & Feature Consolidation Pipeline

This critical section fully operationalizes the **Plant 1 Core Engineering Pipeline**. We implement rigorous data remediation architectures to handle hardware anomalies, execute cross-sectional sensor imputations, relational merges, and continuous weather sequence interpolations before committing the finalized feature matrix to storage.

### ⚙️ Pipeline Execution Rationale:
1. **Blackout Boundary Excision (Task 1):** Permanently truncating the 3-day full communication blackout window (`2020-05-19` to `2020-05-21`) to remove empty telemetric records that would corrupt model training vectors.
2. **Cross-Sectional Median Imputation (Task 2):** Dynamically tracking the operational grid median per timestamp (`plant_dc_median`) to repair anomalous midday inverter dropouts ($0.0\text{ kW}$ under peak radiation).
3. **Relational Influx Synchronization (Task 3):** Committing a relational 1:1 join with meteorological variables ($Ambient\_Temperature$, $Module\_Temperature$, $Irradiation$) followed by localized linear interpolation to seal short-term tracking dropouts.
4. **Production Directory Governance (Task 4):** Validating the operational storage layer and exporting the high-integrity master dataframe with zero index artifacts.

In [ ]:
# ==============================================================================
# SECTION 16: PRODUCTION-GRADE DATA PREPARATION PIPELINE (ENTERPRISE LOGGING)
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Executing comprehensive data remediation and imputation sequences for Plant 1.")

# ------------------------------------------------------------------------------
# TASK 1: Purge Plant 1 Critical Outage Windows (System Blackouts)
# ------------------------------------------------------------------------------
logger.info("Task 1: Commencing physical excision of the 3-day communication blackout timeframe.")
p1_blackout_dates = ['2020-05-19', '2020-05-20', '2020-05-21']
p1_gen_clean = p1_gen_clean[~p1_gen_clean['date_str'].isin(p1_blackout_dates)].copy()

logger.info(f"Task 1 complete. Purged blackout windows. Retained generation matrix rows: {p1_gen_clean.shape[0]:,}")


# ------------------------------------------------------------------------------
# TASK 2: Execute Cross-Sectional Median Imputation for Plant 1 Faulty Inverters
# ------------------------------------------------------------------------------
logger.info("Task 2: Compiling healthy reference frame profile via cross-sectional array medians.")
p1_timestamp_median = p1_gen_clean.groupby('DATE_TIME')['DC_POWER'].median().reset_index()
p1_timestamp_median.columns = ['DATE_TIME', 'healthy_dc_median']

# Left join to inject the baseline reference layer into the localized clean dataframe
p1_gen_clean = pd.merge(p1_gen_clean, p1_timestamp_median, on='DATE_TIME', how='left')

# Isolate the explicit peak-hour midday hardware fault mask specific to Plant 1 (11:00 AM - 03:00 PM)
p1_is_hardware_fault = (
    (p1_gen_clean['time_str'] >= '11:00') & 
    (p1_gen_clean['time_str'] <= '15:00') & 
    (p1_gen_clean['DC_POWER'] == 0.0)
)

p1_faulty_count = p1_gen_clean[p1_is_hardware_fault].shape[0]
logger.info(f"Task 2: Identified {p1_faulty_count:,} peak-hour sub-asset zero-power anomaly points.")

# Overwrite hardware exceptions using the localized healthy cross-sectional median baseline
p1_gen_clean.loc[p1_is_hardware_fault, 'DC_POWER'] = p1_gen_clean.loc[p1_is_hardware_fault, 'healthy_dc_median']
logger.info("Task 2 complete. Anomalous hardware dropouts successfully repaired.")


# ------------------------------------------------------------------------------
# TASK 3: Relational Feature Merge & Climate Metric Interpolation
# ------------------------------------------------------------------------------
logger.info("Task 3: Synchronizing meteorological independent feature matrices via relational join.")
p1_weather_features = p1_weather_clean[['DATE_TIME', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']].copy()

# Left join to preserve all sanitized Plant 1 generation data rows and structure flow chronologically
p1_master_df = pd.merge(p1_gen_clean, p1_weather_features, on='DATE_TIME', how='left').sort_values('DATE_TIME').reset_index(drop=True)

# Smooth over localized minor weather sensor drops using linear interpolation (Cap safety margin at 2 indices)
weather_cols = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
p1_master_df[weather_cols] = p1_master_df[weather_cols].interpolate(method='linear', limit=2)

logger.info("Task 3 complete. Feature merge and environmental sensor smoothing finalized.")


# ------------------------------------------------------------------------------
# TASK 4: Export Clean Engineered Features into Structured Storage Directory
# ------------------------------------------------------------------------------
processed_directory = '../data/processed'
os.makedirs(processed_directory, exist_ok=True)

final_output_path = os.path.join(processed_directory, 'plant1_master_features.csv')
p1_master_df.to_csv(final_output_path, index=False)
logger.info(f"Task 4 complete. Cleaned master array saved to disk storage destination: {final_output_path}")


# ------------------------------------------------------------------------------
# FINAL AUDIT SIGN-OFF
# ------------------------------------------------------------------------------
print("\n" + "="*95)
print(" CRISP-DM STEP 3 AUDIT SIGN-OFF: PLANT 1 PRODUCTION MATRIX DEPLOYED SUCCESSFULLY")
print("===============================================================================================")
print(f"Target Output Storage Destination : {final_output_path}")
print(f"Final Deployed Feature Dimensions : {p1_master_df.shape[0]:,} rows × {p1_master_df.shape[1]} columns")
print("-----------------------------------------------------------------------------------------------")
print("Missing Telemetry Integrity Log (Remaining Null Fields Post-Pipeline Verification):")
print(p1_master_df[weather_cols].isnull().sum().to_string())
print("===============================================================================================")

logger.info("Plant 1 data preparation pipeline closed. Data is highly optimized for leak-free modeling!")

## 🔍 Plant 1: Quality Assurance & Visual Verification Dashboard (Before vs. After Remediation)

To mathematically and visually validate the performance of our cross-sectional imputation pipeline, this section operationalizes a **Quality Assurance (QA) Verification Dashboard**. We isolate a confirmed hardware failure event (Asset `1BY6WEcLGh8j5v7` on `2020-06-14`) and overlay the original corrupted rows directly against the sanitized production array.

### 📐 Strategic Engineering Measures Applied:
1. **Algorithmic Accountability Verification:** Providing explicit side-by-side behavioral tracking to ensure the mathematical median imputation successfully restored the missing peak radiation curve without altering healthy data signatures.
2. **Defensive Isolation Constraints:** Extracting data streams from un-cleansed cache replicas (`raw_staging`) to maintain absolute objectivity during the comparative audit sequence.
3. **Deterministic Coordinate Mapping:** Anchoring time tick indices via structured integer vectors (`np.arange`), completely neutralizing matplotlib time-string compression anomalies and user warning outputs.

In [ ]:
# ==============================================================================
# SECTION 17: PIPELINE QUALITY ASSURANCE & VISUAL VERIFICATION DASHBOARD
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initializing pipeline data remediation visual quality assurance audit.")

test_date = '2020-06-14'
test_culprit = '1BY6WEcLGh8j5v7'

# 2. Replicate the un-cleansed tracking framework to establish the raw baseline
raw_staging = plant1_gen_df.copy()
raw_staging['DATE_TIME'] = pd.to_datetime(raw_staging['DATE_TIME'])
raw_staging['date_str'] = raw_staging['DATE_TIME'].dt.strftime('%Y-%m-%d')
raw_staging['time_str'] = raw_staging['DATE_TIME'].dt.strftime('%H:%M')

# 3. Isolate the target anomaly slice before and after data engineering remediation
raw_day = raw_staging[(raw_staging['date_str'] == test_date) & (raw_staging['SOURCE_KEY'] == test_culprit)].sort_values('DATE_TIME')
clean_day = p1_master_df[(p1_master_df['date_str'] == test_date) & (p1_master_df['SOURCE_KEY'] == test_culprit)].sort_values('DATE_TIME')

# 4. Initialize Visual Verification Plot Canvas
plt.figure(figsize=(14, 6))

# Plot the raw un-remediated series in crimson red to highlight the severe drop footprint
plt.plot(raw_day['time_str'], raw_day['DC_POWER'], color='crimson', marker='o', alpha=0.5, label='Original Corrupted Data (Raw 0.0 kW Drops)')

# Plot the sanitized master feature series in royal blue to verify the median repair vector
plt.plot(clean_day['time_str'], clean_day['DC_POWER'], color='royalblue', marker='.', linestyle='--', lw=2, label='Sanitized Data (After Cross-Sectional Imputation)')

# 5. FIX: Apply Explicit Anchored Coordinate Ticks to Prevent Timeline Displacement and UserWarnings
ax = plt.gca()
sample_timeline = sorted(raw_day['time_str'].unique())
target_indices = np.arange(0, len(sample_timeline), 8)  # Space major ticks every 2 hours (8 intervals of 15-min logs)
target_labels = [sample_timeline[idx] for idx in target_indices]

ax.set_xticks(target_indices)
ax.set_xticklabels(target_labels, rotation=45, ha='right', fontsize=9)

plt.title(f"Plant 1: Preprocessing Pipeline Quality Assurance Sign-Off (Asset: {test_culprit})", fontsize=14, fontweight='bold')
plt.title(f"Visual Verification of Remediation Success for Inverter {test_culprit} ({test_date})", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day (24-Hour Operational Cycle)", fontsize=11)
plt.ylabel("Direct Current Power Output (kW)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

logger.info("Pipeline visual verification audit completed successfully. Preprocessing logic signed off.")

# 🤖 Plant 1: Prospective Time-Series Partitioning & LightGBM Machine Learning Pipeline

This operational milestone implements a rigorous **Chronological Holdout Split** (strictly avoiding random data shuffling to eliminate target leakage) and fits an enterprise-grade **LightGBM Regressor** to project prospective power forecasting vectors into an unseen future timeline.

### 📐 Validation Architecture Strategy:
1. **Historical Training Base (05/15 ~ 06/11):** Establishes the core atmospheric causality matrix mapping independent features ($Ambient\_Temperature$, $Module\_Temperature$, $Irradiation$) to the physical $DC\_POWER$ generation target.
2. **Prospective Future Test Window (06/12 ~ 06/17):** Functions as a completely blind evaluation timeline, simulating how the pipeline handles real-time streaming operations on unseen future calendar slots.
3. **Enterprise Structured Logging Engine:** Replaces casual terminal prints with a synchronized, production-grade tracking standard to record optimization checkpoints cleanly.

In [ ]:
# ==============================================================================
# SECTION 18: TIME-SERIES PARTITION & LIGHTGBM EVALUATION (ENTERPRISE LOGGING)
# ==============================================================================

# 1. Initialize Pipeline-Level Logging Configuration
logger = logging.getLogger("IngestionPipeline")
logger.info("Initializing chronological timeline partitioning sequence for Plant 1 arrays.")

# Synchronize datetime data types and string indexing variables
p1_master_df['DATE_TIME'] = pd.to_datetime(p1_master_df['DATE_TIME'])

# Rely strictly on our synchronized local lowercase date tracker format string
if 'date_str' not in p1_master_df.columns:
    p1_master_df['date_str'] = p1_master_df['DATE_TIME'].dt.strftime('%Y-%m-%d')
if 'time_str' not in p1_master_df.columns:
    p1_master_df['time_str'] = p1_master_df['DATE_TIME'].dt.strftime('%H:%M')

X_master = p1_master_df[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']]
y_master = p1_master_df['DC_POWER']
dates = p1_master_df['date_str']

# 2. Enforce strict chronological partitioning to eliminate data leakage
# Train interval: Base Start (05/15) ~ 06/11 | Evaluation interval: Unseen Future (06/12 ~ 06/17)
train_mask = dates <= '2020-06-11'
test_mask  = (dates >= '2020-06-12') & (dates <= '2020-06-17')

X_train, y_train = X_master[train_mask], y_master[train_mask]
X_test, y_test   = X_master[test_mask], y_master[test_mask]

# 3. Pipeline Partition Structure Verification Audit
print("\n" + "="*80)
print(" CHRONOLOGICAL TIMELINE PARTITION INTEGRITY REPORT")
print("==============================================================================")
print(f"Training Knowledge Base (05/15 ~ 06/11) : {X_train.shape[0]:,} rows")
print(f"Evaluation Test Window  (06/12 ~ 06/17) : {X_test.shape[0]:,} rows")
print("==============================================================================\n")

# 4. Fit Enterprise-Grade LightGBM Regressor on Training Baseline
logger.info("Training LightGBM forecasting engine on operational training historical rows...")
verify_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
verify_model.fit(X_train, y_train)

# 5. Execute Prospective Forecasting Over the Blind Test Window
logger.info("Projecting prospective power vectors over the unseen future test window.")
y_pred = verify_model.predict(X_test)

# 6. Evaluate Model Performance Metrics Using Global Standards
rmse_eval = np.sqrt(mean_squared_error(y_test, y_pred))
r2_eval = r2_score(y_test, y_pred)

# 7. Render Final Production Score Sign-Off
print("\n" + "="*80)
print(" FINAL OPERATIONAL PERFORMANCE SIGN-OFF: PLANT 1 FORECASTING REPORT")
print("==============================================================================")
print(f"Plant 1 Future Validation RMSE     : {rmse_eval:.4f} kW")
print(f"Plant 1 Future Validation R2 Score : {r2_eval:.4f} ({r2_eval*100:.2f}% Variance Match)")
print("==============================================================================")
logger.info("Modeling pipeline evaluation sequence completed successfully.")
print("==============================================================================")

## 📊 Plant 1: Prospective Trend Tracking & Diurnal Cycle Validation Dashboard

To visually inspect the algorithmic generalizability and capture physical curve tracking accuracies, this section operationalizes a **Dual-Horizon Prospective Validation Dashboard**. We evaluate continuous multi-day trend vectors alongside an isolated 24-hour diurnal profile focus (`2020-06-17`) over completely unseen future calendar blocks.

### 📐 Strategic Engineering Layout Rules Applied:
1. **Defensive Namespace Safeguards:** Deploying a conditional local cache intercept statement to automatically reconstruct and fit missing training states if notebook memory variables are flushed.
2. **Macro-To-Micro Diagnostic Continuum:** Mapping a continuous multi-day scatter-to-line trajectory (Chart 1) followed by a granular day-level operational cycle wave (Chart 2) to audit systemic bias.
3. **Deterministic Coordinate Ticker Anchoring:** Replacing fragile dynamic tick gathering layers with explicit index range allocations (`np.arange`), ensuring 24-hour military text indicators scale correctly across all localized python environments.

In [ ]:
# ==============================================================================
# SECTION 19: PROSPECTIVE TREND TRACKING & DIURNAL CYCLE VALIDATION DASHBOARD
# ==============================================================================

# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initializing multi-horizon prospective forecasting tracking dashboards.")

# [SAFEGUARD] If the notebook memory cache was flushed via a kernel restart, 
# dynamically reconstruct the chronological masks, fit the model, and generate predictions.
if 'y_pred' not in locals() or 'test_mask' not in locals():
    logger.warning("Notebook memory cache empty. Dynamically synchronizing upstream pipeline elements...")
    
    p1_master_df['DATE_TIME'] = pd.to_datetime(p1_master_df['DATE_TIME'])
    
    if 'date_str' not in p1_master_df.columns:
        p1_master_df['date_str'] = p1_master_df['DATE_TIME'].dt.strftime('%Y-%m-%d')
    if 'time_str' not in p1_master_df.columns:
        p1_master_df['time_str'] = p1_master_df['DATE_TIME'].dt.strftime('%H:%M')

    X_master = p1_master_df[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']]
    y_master = p1_master_df['DC_POWER']
    dates = p1_master_df['date_str']

    train_mask = dates <= '2020-06-11'
    test_mask  = (dates >= '2020-06-12') & (dates <= '2020-06-17')

    X_train, y_train = X_master[train_mask], y_master[train_mask]
    X_test, y_test   = X_master[test_mask], y_master[test_mask]

    fallback_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    fallback_model.fit(X_train, y_train)
    y_pred = fallback_model.predict(X_test)
    logger.info("Upstream state vectors securely restored. Fallback model execution closed.")

# 2. Assemble Staging Validation Frame Over the Blind Evaluation Window
eval_df = p1_master_df[test_mask].copy().sort_values('DATE_TIME')
eval_df['pred_dc_power'] = y_pred


# ------------------------------------------------------------------------------
# CHART 1: [Multi-Day Trend] Macro Tracking Across Consecutive Dates
# ------------------------------------------------------------------------------
logger.info("Rendering Chart 1: multi-day macro continuous prospective forecasting trends.")
plt.figure(figsize=(18, 6))

# Plot the real historical truth as crimson scatter points
plt.scatter(eval_df['DATE_TIME'], eval_df['DC_POWER'], 
            color='crimson', alpha=0.4, s=12, label='Actual Output (Real Reality)')

# Plot the model forecasts as a continuous, smooth royalblue tracking line
plt.plot(eval_df['DATE_TIME'], eval_df['pred_dc_power'], 
         color='royalblue', lw=2, alpha=0.9, label='LightGBM Multi-Day Trend Forecast')

# Format the X-axis tick distribution cleanly into day intervals without anomalies
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
plt.gca().xaxis.set_major_locator(mdates.DayLocator())

plt.title("Plant 1: 6-Day Continuous Power Generation Trend & Tracking (June 12 - June 17)", fontsize=14, fontweight='bold')
plt.xlabel("Timeline Target Operational Window (Date Calendar)", fontsize=11)
plt.ylabel("Direct Current Power Output (kW)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------------------
# CHART 2: [24-Hour Diurnal Profile] Micro Physical Curve Verification
# ------------------------------------------------------------------------------
target_focus_date = '2020-06-17'
logger.info(f"Rendering Chart 2: 24-hour diurnal micro physical curve verification for focus date: {target_focus_date}")

final_day_df = eval_df[eval_df['date_str'] == target_focus_date].sort_values('DATE_TIME')

plt.figure(figsize=(15, 6))

# Overlay real metrics dots with the model's predictive tracking horizon line
plt.scatter(final_day_df['time_str'], final_day_df['DC_POWER'], 
            color='crimson', alpha=0.6, s=35, label='Actual Output (Real Reality)')
plt.plot(final_day_df['time_str'], final_day_df['pred_dc_power'], 
         color='royalblue', lw=2.5, label='LightGBM 24-Hour Micro Forecast Track')

# FIX: Secure explicit anchored ticks matrix formatting instead of dynamic ax.get_xticks() intercepting
ax = plt.gca()
sample_timeline = sorted(final_day_df['time_str'].unique())
target_indices = np.arange(0, len(sample_timeline), 16)  # Space major ticks every 4 hours (16 steps of 15-min intervals)
target_labels = [sample_timeline[idx] for idx in target_indices]

ax.set_xticks(target_indices)
ax.set_xticklabels(target_labels, rotation=45, ha='right', fontsize=9)

plt.title(f"Plant 1: 24-Hour Diurnal Generation Curve vs Reality (Focus Date: {target_focus_date})", fontsize=14, fontweight='bold')
plt.xlabel("Time of Day (24-Hour Operational Cycle)", fontsize=11)
plt.ylabel("Direct Current Power Output (kW)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)
plt.tight_layout()
plt.show()

logger.info("Prospective visual verification dashboards compiled successfully.")

## 📊 Plant 1: Model Interpretability & Statistical Feature Auditing Dashboard

To establish rigorous algorithmic interpretability and validate environmental causality, this section constructs a **Dual-Perspective Feature Audit Dashboard**. We cross-examine the tree-splitting operational footprint of our trained LightGBM model (Chart 1) against the classical linear Pearson correlation space (Chart 2) to ensure physical domain alignment.

### 📐 Strategic Engineering Layout Rules Applied:
1. **Algorithmic Explainability (XAI):** Extracting structural feature importance scores to reveal how the tree ensemble weights independent variables under non-linear conditions.
2. **Linear Causality Cross-Checking:** Mapping a symmetric Pearson correlation heatmap ($vmin=-1.0, vmax=1.0$) to audit target-to-feature colinearities without floating anomalies.
3. **Defensive Workspace Recovery:** Embedding a conditional local cache fallback statement to smoothly rebuild variables and re-fit parameters if notebook memory states are lost.

In [ ]:
# ==============================================================================
# SECTION 20: MODEL INTERPRETABILITY & STATISTICAL FEATURE AUDITING DASHBOARD
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Assembling 1x2 explainable AI feature importance and correlation matrix diagnostics.")

# [SAFEGUARD] If the notebook memory cache was flushed via a kernel restart, 
# dynamically reconstruct the dataset split and re-fit the baseline model.
if 'verify_model' not in locals() or 'X_train' not in locals():
    logger.warning("Notebook memory cache empty. Dynamically synchronizing upstream training elements...")
    
    p1_master_df['DATE_TIME'] = pd.to_datetime(p1_master_df['DATE_TIME'])
    
    if 'date_str' not in p1_master_df.columns:
        p1_master_df['date_str'] = p1_master_df['DATE_TIME'].dt.strftime('%Y-%m-%d')

    X_master = p1_master_df[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']]
    y_master = p1_master_df['DC_POWER']
    dates = p1_master_df['date_str']

    train_mask = dates <= '2020-06-11'
    X_train, y_train = X_master[train_mask], y_master[train_mask]

    verify_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    verify_model.fit(X_train, y_train)
    logger.info("Upstream model vectors securely restored. Fallback model execution closed.")

# 2. Initialize a 1x2 Dual-Plot Visualization Grid Canvas
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ------------------------------------------------------------------------------
# CHART 1: LightGBM Feature Importance (AI's Operational Choice)
# ------------------------------------------------------------------------------
# Extract native node-splitting metrics from the verified LightGBM estimator
importance = verify_model.feature_importances_
feature_names = X_train.columns

# Structure feature records into a sorted visualization dataframe
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=True)

# Render a horizontal bar plot showing relative feature contributions
axes[0].barh(importance_df['feature'], importance_df['importance'], color='teal', edgecolor='black', alpha=0.8)
axes[0].set_title("1. LightGBM Feature Importance (AI's Choice)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Importance Score (Number of tree splits)", fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.3)


# ------------------------------------------------------------------------------
# CHART 2: Correlation Matrix Heatmap (Surface-Level Human Perspective)
# ------------------------------------------------------------------------------
# Calculate linear Pearson correlation coefficients across target numeric vectors
analysis_cols = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'DC_POWER']
correlation_matrix = p1_master_df[analysis_cols].corr()

# Draw a stylized thematic heatmap matrix with explicit value annotations
sns.heatmap(
    correlation_matrix, 
    annot=True, 
    cmap='coolwarm', 
    fmt=".3f", 
    linewidths=0.5, 
    vmin=-1, vmax=1,
    ax=axes[1]
)
axes[1].set_title("2. Feature Correlation Matrix (Human View)", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

logger.info("Explainable AI diagnostic validation dashboard generated successfully.")

## 📊 Plant 1: Alternating Current ($AC$) Conversion & Daily Integrated Volumetric Energy Audit

This final verification milestone bridges the gap between instantaneous telemetric power forecasts ($kW$) and daily commercial energy volume outputs ($kWh$). We apply a standardized inverter physical efficiency factor ($92\%$) to simulate Alternating Current ($AC$) power, execute a time-series numerical integration over 15-minute intervals ($\Delta t = 0.25 \text{ hours}$), and audit daily aggregated target errors over the blind test window.

### 📐 Engineering & Validation Measures Applied:
1. **Thermodynamic Efficiency Simulation:** Multiplying Direct Current metrics by an industry-standard inverter conversion coefficient ($\eta = 0.92$) to synthesize grid-side energy profiles safely.
2. **Riemann Sum Numerical Integration:** Compounding high-frequency instantaneous power tracks into true total volumetric energy capacities ($\text{kW} \times \text{0.25 hours} = \text{kWh}$).
3. **Financial Error Delta Tracking:** Computing explicit variance columns (`error_delta_kwh`) to reveal macroscopic over/under-forecasting bias for downstream grid trading or asset storage strategies.
4. **Structured Format Serialization:** Enforcing explicit string format masks and lowercase snake_case naming variables across the compiled report ledger.

In [ ]:
# ==============================================================================
# SECTION 21: INDUSTRIAL AC CONVERSION & VOLUMETRIC DAILY YIELD AUDIT
# ==============================================================================
# 1. Access the Pipeline-Level Logger Instance
logger = logging.getLogger("IngestionPipeline")
logger.info("Initializing high-precision AC conversion and numeric integration energy auditing pipelines.")

# [SAFEGUARD] If the notebook memory cache was flushed via a kernel restart, 
# dynamically reconstruct chronological masks, re-fit the model, and generate predictions.
if 'y_pred' not in locals() or 'test_mask' not in locals():
    logger.warning("Notebook memory cache empty. Dynamically synchronizing upstream pipeline elements...")
    
    p1_master_df['DATE_TIME'] = pd.to_datetime(p1_master_df['DATE_TIME'])
    
    if 'date_str' not in p1_master_df.columns:
        p1_master_df['date_str'] = p1_master_df['DATE_TIME'].dt.strftime('%Y-%m-%d')

    X_master = p1_master_df[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']]
    y_master = p1_master_df['DC_POWER']
    dates = p1_master_df['date_str']

    train_mask = dates <= '2020-06-11'
    test_mask  = (dates >= '2020-06-12') & (dates <= '2020-06-17')

    X_train, y_train = X_master[train_mask], y_master[train_mask]
    X_test, y_test   = X_master[test_mask], y_master[test_mask]

    fallback_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
    fallback_model.fit(X_train, y_train)
    y_pred = fallback_model.predict(X_test)
    logger.info("Upstream state vectors securely restored. Fallback model execution closed.")

# 2. Extract Validation Baseline Matrix for the Prospective Target Window
eval_df = p1_master_df[test_mask].copy().sort_values('DATE_TIME')
eval_df['pred_dc_power'] = y_pred

# 3. Apply Standard Inverter Efficiency Constraints (AC Power Simulation Layer)
INVERTER_EFFICIENCY = 0.92
eval_df['actual_ac_power_sim'] = eval_df['DC_POWER'] * INVERTER_EFFICIENCY
eval_df['pred_ac_power_sim']   = eval_df['pred_dc_power'] * INVERTER_EFFICIENCY

# 4. Riemann Sum Numerical Integration (kW * 0.25 Hours Time Index = Total kWh Energy Volume)
eval_df['actual_kwh'] = eval_df['actual_ac_power_sim'] * 0.25
eval_df['pred_kwh']   = eval_df['pred_ac_power_sim'] * 0.25

# Group by the synchronized lowercase string column to calculate daily accumulated energy capacities
daily_yield_report = eval_df.groupby('date_str')[['actual_kwh', 'pred_kwh']].sum().reset_index()

# Flatten headers into strict lowercase snake_case compliance styles for production pipelines
daily_yield_report.columns = ['date', 'actual_daily_yield_kwh', 'predicted_daily_yield_kwh']
daily_yield_report['error_delta_kwh'] = daily_yield_report['predicted_daily_yield_kwh'] - daily_yield_report['actual_daily_yield_kwh']

# 5. Render Financial & Operational Deployed Integrity Ledger Reports
print("\n" + "="*85)
print(" FINANCIAL & OPERATIONAL PRODUCTION LEDGER REPORT: DAILY INDUSTRIAL ENERGY CAPACITIES")
print("=====================================================================================")
print(daily_yield_report.to_string(index=False, formatters={
    'actual_daily_yield_kwh': '{:,.2f}'.format,
    'predicted_daily_yield_kwh': '{:,.2f}'.format,
    'error_delta_kwh': '{:+,.2f}'.format
}))
print("=====================================================================================\n")

# 6. Initialize Comparative Clustered Bar Dashboard Framework
plt.figure(figsize=(12, 6))
x_axis = np.arange(len(daily_yield_report['date']))

# Map reality metrics as crimson bars and AI prospective forecasts as royalblue bars side-by-side
plt.bar(x_axis - 0.2, daily_yield_report['actual_daily_yield_kwh'], width=0.4, color='crimson', alpha=0.7, label='Actual Yield (Physical Reality)')
plt.bar(x_axis + 0.2, daily_yield_report['predicted_daily_yield_kwh'], width=0.4, color='royalblue', alpha=0.7, label='Predicted Yield (LightGBM Pipeline Forecast)')

plt.xticks(x_axis, daily_yield_report['date'])
plt.title("Plant 1: Multi-Day Total Daily Integrated Yield Verification (Actual vs AI Forecast)", fontsize=14, fontweight='bold')
plt.xlabel("Target Future Prospective Forecast Date", fontsize=11)
plt.ylabel("Consolidated Total Daily Energy Volume (kWh)", fontsize=11)
plt.grid(True, axis='y', linestyle='--', alpha=0.3)
plt.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

logger.info("Daily integrated energy metrics verified and dashboard graphs outputted successfully.")